In [5]:
import httpx
import time
from IPython.display import display, Markdown

OLLAMA_URL = "http://localhost:11434/api/chat"

MODELS = {
    "Llama 3.2 (Meta)": "llama3.2:3b",
    "Gemma 2 (Google)": "gemma2:2b",
    "Phi 3 (Microsoft)": "phi3:mini"
}

TEST_PROMPT = """I am 25 years old, earn $4000 AUD per month, 
spend $1500 AUD per month on expenses, have no debt, 
want to save for a house, and have medium risk tolerance. 
Please give me a personalized financial plan."""

SYSTEM = """You are a professional personal financial assistant. 
Provide structured, personalized financial advice."""

def query_model(model_name: str, prompt: str) -> tuple:
    start = time.time()
    response = httpx.post(OLLAMA_URL, json={
        "model": model_name,
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt}
        ],
        "stream": False
    }, timeout=120.0)
    elapsed = time.time() - start
    data = response.json()
    return data["message"]["content"], round(elapsed, 2)

In [6]:
results = {}

for label, model in MODELS.items():
    print(f"\n{'='*60}")
    print(f"MODEL: {label}")
    print(f"{'='*60}")
    print(f"Querying {model}...")
    
    try:
        response, elapsed = query_model(model, TEST_PROMPT)
        results[label] = {
            "response": response,
            "time": elapsed,
            "words": len(response.split())
        }
        print(f"Response Time: {elapsed}s")
        print(f"Word Count: {len(response.split())}")
        print(f"\nResponse:\n{response}")
    except Exception as e:
        print(f"Error: {e}")


MODEL: Llama 3.2 (Meta)
Querying llama3.2:3b...
Response Time: 16.27s
Word Count: 438

Response:
Congratulations on taking the first step towards securing your financial future! Based on your income, expenses, and goals, I'll provide a personalized financial plan tailored to your needs.

**Income and Expenses:**

* Gross income: $4,000 AUD per month
* Net income (after taxes): approximately $3,200 AUD per month (assuming 25% tax bracket)
* Fixed expenses:
	+ Essential expenses (housing, food, transportation, utilities): $1,500 AUD per month
	+ Savings and debt repayment: aim to save at least 20% of net income ($640 AUD per month)
	+ Other discretionary spending (entertainment, hobbies): variable

**Financial Goals:**

* Save for a house: aim to achieve a 20% deposit within the next 12-18 months
* Medium risk tolerance: we'll focus on investing in low-to-moderate risk assets to balance potential returns with stability

**Recommendations:**

1. **Essential Expenses:** Ensure you're cove

# 📊 Comparison Summary"

In [7]:

table = "| Model | Company | Response Time | Word Count |\n"
table += "|-------|---------|--------------|------------|\n"

for label, data in results.items():
    if "error" not in data:
        company = label.split("(")[1].replace(")", "")
        table += f"| {label.split('(')[0]} | {company} | {data['time']}s | {data['words']} |\n"

display(Markdown(table))


| Model | Company | Response Time | Word Count |
|-------|---------|--------------|------------|
| Llama 3.2  | Meta | 16.27s | 438 |
| Gemma 2  | Google | 13.94s | 495 |
| Phi 3  | Microsoft | 27.54s | 653 |



## Analysis:


All three models produced usable financial plans, but there were clear differences worth noting. Gemma 2 was the fastest at 13.86 seconds and gave a clean, well-structured response, though it was the least detailed of the three. Llama 3.2 struck a good balance — reasonable response time and a solid level of detail, including Australian-specific references like Superannuation. Phi 3 produced the longest response by far at 835 words, but the extra length didn't always mean better quality — some sections felt repetitive and it was the slowest at 34 seconds, which would noticeably affect user experience in a live chat setting.

# Selected Model: Llama 3.2 (Meta)

Llama 3.2 was selected for the production application. It offered the best balance between response quality, structure, and speed. The output was detailed enough to be genuinely useful while remaining readable, and it referenced Australian financial context naturally without being prompted to do so.


